# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json

**Note**: All dataset entities (record sets, fields, columns) are referenced by their `@id` per Croissant standards.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant JSON-LD URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Name: {metadata.name}\n\nDescription: {metadata.description}\n")
print(f"Version: {metadata.version}")
print(f"Published: {metadata.date_published}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All Croissant entities are referenced by `@id` for clarity and reproducibility.

Let's print all record sets available in the dataset and their field schema (by `@id`).

In [ ]:
# List available record sets and their fields by `@id`

from pprint import pprint
record_sets = list(dataset.record_sets)  # a list of mlcroissant.RecordSet
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description or 'No description'}")
    print("  Fields (@id : type):")
    for field in rs.fields:
        print(f"    - {field.id} : {field.data_type}")
    print("\n")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use record set and field `@id`s identified above.

We'll demonstrate with the main data record set, likely containing survey responses (often named something like `cr:RecordSet/survey` or similar). **Change the `record_set_id` below if needed, based on the output above.**

In [ ]:
# Extract available record set @ids for convenience
record_set_ids = [rs.id for rs in record_sets]
pprint(record_set_ids)

# For this dataset, we'll use the first (and often the main) record set.
record_set_id = record_set_ids[0]  # adjust if necessary based on printed output above

print(f"Loading records from record set: {record_set_id}")

# Load records for each available record set and put them in a dictionary of DataFrames
dataframes = {}
for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    df = pd.DataFrame(records)
    dataframes[rsid] = df
    print(f"Records loaded for {rsid}: {len(df)} entries, {len(df.columns)} columns")
    if rsid == record_set_id:
        print(f"Column names for {rsid}:")
        pprint(list(df.columns))
        display(df.head())

## 4. Exploratory Data Analysis (EDA)
Let's apply some common data processing steps: filtering by field, normalization, detecting simple outliers, and grouping.

**Replace field `@id`s below as appropriate** according to what's available in your data columns.

In [ ]:
# Identify a numeric field for exploration (e.g., GAD-7, PHQ-9, PSQ total scores - adapt to the output column names)
# For demonstration, let's pick the first numeric column (search for one containing numeric data)
df = dataframes[record_set_id]
numeric_field_candidate = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_candidate = col
        break

if numeric_field_candidate is None:
    raise ValueError("No numeric field detected for analysis. Check your data.")

numeric_field = numeric_field_candidate  # set as our field of interest
print(f"Selecting numeric field for EDA: {numeric_field}")

# Example: filter values > some threshold (we use the mean for demonstration, or change as needed)
threshold = df[numeric_field].mean()
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f}: {len(filtered_df)} records.")
display(filtered_df.head())

# Normalize the numeric field (z-score)
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"First few records after normalization of {numeric_field}:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a categorical field, e.g., by 'gender' or 'marital_status' if present. Pick the first categorical column as an example.
categorical_field = None
for col in df.columns:
    if pd.api.types.is_string_dtype(df[col]) and col != numeric_field:
        categorical_field = col
        break

if categorical_field:
    grouped_df = filtered_df.groupby(categorical_field)[numeric_field].mean().reset_index()
    print(f"Mean {numeric_field} grouped by {categorical_field}:")
    display(grouped_df)
else:
    print("No suitable categorical field found for grouping.")

## 5. Visualization
Let's visualize the distribution of the numeric field selected above, and its relationship with the selected categorical field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric_field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# Boxplot by categorization if applicable
if categorical_field:
    plt.figure(figsize=(10, 4))
    sns.boxplot(x=df[categorical_field], y=df[numeric_field])
    plt.title(f"{numeric_field} by {categorical_field}")
    plt.xticks(rotation=45)
    plt.show()
else:
    print("No categorical field selected for boxplot.")

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² Mental Health Survey dataset from Kilifi County, Kenya, using `mlcroissant`. We examined record sets and fields via their Croissant `@id`, loaded the survey data into a DataFrame, and performed basic exploratory data analysis including filtering, normalization, grouping, and visualization.

- The data provides insight into mental health indicators and demographics for Kilifi county.
- Using Croissant `@id` referencing ensures reproducibility and semantic clarity.
- This notebook can be extended with domain-specific analyses relevant to mental health, public health interventions, or AI-readiness assessments relevant to African data infrastructure.